# Building our model using Catboost

## Importing packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
%pip install catboost
from catboost import CatBoostClassifier, Pool

## Importing our data

In [ ]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/processed/cleaned_train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/processed/cleaned_train.csv'
    },
    'test': {
        'local': '../data/processed/cleaned_test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/processed/cleaned_test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }
    
}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

## Setting up our columns

In [ ]:
target_col = 'cost_category'
id_col = 'Tour_ID'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']

# Map string labels to numeric integers (0 to 5) for multi-class training
class_to_idx = {cls_name: i for i, cls_name in enumerate(target_classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(target_classes)}
train['target'] = train[target_col].map(class_to_idx)

## Categorical Feature Specification

In [ ]:
# Define all string/categorical columns for CatBoost
cat_features = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity', 
    'info_source', 'tour_arrangement', 'package_transport_int', 
    'package_accomodation', 'package_food', 'package_transport_tz', 
    'package_sightseeing', 'package_guided_tour', 'package_insurance', 'first_trip_tz'
]
train_df = train.copy()
test_df = test.copy()

# Ensure categorical columns are strings (CatBoost requirement for cat_features)
for col in cat_features:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

features = [col for col in train_df.columns if col not in [id_col, target_col, 'target']]

X = train_df[features]
y = train_df['target']
X_test = test_df[features]

## Stratified 5-Fold Cross-Validation with CatBoost

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(train_df), len(target_classes)))
test_preds = np.zeros((len(test_df), len(target_classes)))

test_pool = Pool(X_test, cat_features=cat_features)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1} ---")
    
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_va, y_va, cat_features=cat_features)
    
    model = CatBoostClassifier(
        iterations=1200,
        learning_rate=0.04,
        depth=6,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        random_seed=42,
        task_type='CPU', # Change to 'GPU' if running with CUDA
        verbose=200
    )
    
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=100,
        use_best_model=True
    )
    
    # Store Out-of-Fold predictions and test predictions
    oof_preds[val_idx] = model.predict_proba(val_pool)
    test_preds += model.predict_proba(test_pool) / skf.n_splits

# Overall Out-Of-Fold Multi-Class Log Loss Metric
cv_log_loss = log_loss(y, oof_preds)
print(f"\n==========================================")
print(f"Overall OOF Log Loss: {cv_log_loss:.5f}")
print(f"==========================================")

## CatBoost Submission

In [ ]:
submission = pd.DataFrame(test_preds, columns=[idx_to_class[i] for i in range(len(target_classes))])
submission.insert(0, id_col, test_df[id_col])

# Reorder columns to strictly match SampleSubmission.csv
target_order = ['Tour_ID', 'High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']
submission = submission[target_order]

submission.to_csv('catboost_submission.csv', index=False)
print("Saved predictions to catboost_submission.csv")